<a href="https://colab.research.google.com/github/USUARIO/REPOSITORIO/blob/main/notebooks/gee_data_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualización genérica de datos geoespaciales de Google Earth Engine (GEE)

Notebook **modular, reproducible y reutilizable** para consultar, procesar y visualizar datasets del catálogo de Google Earth Engine con `earthengine-api` y `geemap`.

La idea central: **cambias unos pocos parámetros** (dataset, banda, período, área de estudio, visualización) y obtienes el mapa, las estadísticas y, opcionalmente, la exportación, **sin tocar la lógica principal**.

> **Antes de empezar:** necesitas un proyecto de Google Cloud registrado para Earth Engine. Si nunca has usado GEE, sigue primero la guía [`GOOGLE_CLOUD_SETUP.md`](../GOOGLE_CLOUD_SETUP.md) del repositorio.

> **Sobre el botón “Open in Colab”:** apunta a `https://colab.research.google.com/github/USUARIO/REPOSITORIO/blob/main/notebooks/gee_data_visualization.ipynb`. Reemplaza `USUARIO` y `REPOSITORIO` por tu usuario y repositorio de GitHub (ver la sección “Abrir en Google Colab” del `README.md`).

## Tabla de contenidos

1. [Requisitos previos](#1-requisitos-previos)
2. [Instrucciones de uso](#2-instrucciones-de-uso)
3. [Arquitectura: servidor vs. cliente](#3-arquitectura-servidor-vs-cliente)
4. [Instalación de dependencias](#4-instalacion-de-dependencias)
5. [Configuración global (parámetros editables)](#5-configuracion-global)
6. [Conexión con Earth Engine](#6-conexion-con-earth-engine)
7. [Módulos del código](#7-modulos-del-codigo)
   - 7.1 Utilidades y excepciones · 7.2 Área de estudio · 7.3 Carga de datos · 7.4 Procesamiento · 7.5 Visualización · 7.6 Función principal `visualize_gee_dataset()` · 7.7 Exportación · 7.8 Registro de datasets
8. [Definición del área de estudio (AOI)](#8-area-de-estudio)
9. [Ejemplo 1 — Cobertura del suelo (ESA WorldCover)](#9-ejemplo-1-cobertura-del-suelo)
10. [Ejemplo 2 — Modelo Digital de Elevación (SRTM)](#10-ejemplo-2-dem)
11. [Ejemplo 3 — Precipitación (CHIRPS Daily)](#11-ejemplo-3-precipitacion)
12. [Ejemplo 4 — Temperatura (ERA5-Land Daily Aggregates)](#12-ejemplo-4-temperatura)
13. [Mapa final combinado y comparación](#13-mapa-final-combinado)
14. [Exportación (opcional)](#14-exportacion)
15. [Cómo agregar un dataset nuevo](#15-como-agregar-un-dataset-nuevo)
16. [Problemas frecuentes](#16-problemas-frecuentes)
17. [Rendimiento y buenas prácticas](#17-rendimiento-y-buenas-practicas)
18. [Referencias](#18-referencias)

<a id="1-requisitos-previos"></a>
## 1. Requisitos previos

| Requisito | Detalle |
|---|---|
| Cuenta de Google | Personal o institucional. |
| Proyecto de Google Cloud | Con la **Earth Engine API habilitada** y **registrado para Earth Engine** (uso no comercial o comercial). Ver `GOOGLE_CLOUD_SETUP.md`. |
| Permisos IAM | Mínimo: `roles/serviceusage.serviceUsageConsumer` + `roles/earthengine.viewer` en el proyecto (o ser Owner/Editor). |
| Entorno | Google Colab (recomendado) o Python ≥ 3.10 local con Jupyter. |
| Paquetes | `earthengine-api`, `geemap`, `pandas` (ver `requirements.txt`). |

**No se requieren** cuentas de servicio, claves JSON ni tokens para el uso interactivo. **Nunca** subas credenciales al repositorio.

<a id="2-instrucciones-de-uso"></a>
## 2. Instrucciones de uso

1. Ejecuta la celda de **instalación** (sección 4).
2. En la **configuración global** (sección 5) cambia como mínimo `PROJECT_ID`. Opcionalmente cambia el AOI y las fechas.
3. Ejecuta la **conexión** (sección 6). En Colab se abrirá una ventana para autorizar tu cuenta de Google.
4. Ejecuta las celdas de **módulos** (sección 7). Solo definen funciones: no consumen cuota.
5. Corre los **ejemplos** (secciones 9–13) en orden. Cada uno muestra un mapa interactivo y estadísticas.
6. Para usar otro dataset, llama a `visualize_gee_dataset(...)` con sus parámetros o agrégalo al `DATASET_REGISTRY` (sección 15).

Atajo: *Entorno de ejecución → Ejecutar todas* funciona una vez configurado `PROJECT_ID`.

<a id="3-arquitectura-servidor-vs-cliente"></a>
## 3. Arquitectura: servidor vs. cliente

El código está separado en capas:

```
Configuración ──► Conexión ──► Carga ──► Procesamiento ──► Visualización ──► Exportación (opcional)
 (sección 5)      (6)          (7.3)       (7.4)              (7.5–7.6)         (7.7)
```

Un punto clave de Earth Engine: **los objetos `ee.*` son recetas, no datos**. Cuando escribes `ee.ImageCollection(...).filterDate(...).sum()`, Python solo arma un grafo de operaciones; el cómputo ocurre **en los servidores de Google** cuando algo pide un resultado.

| Operación | ¿Dónde se ejecuta? | Comentario |
|---|---|---|
| Construir `ee.Image`, `ee.Geometry`, filtros, `.sum()`, `.mean()`, `.clip()`, `.subtract()` | **Servidor** (diferido) | No transfiere píxeles; solo define el grafo. |
| Teselas del mapa (`Map.add_layer`) | **Servidor** | GEE renderiza teselas PNG bajo demanda según el zoom. No se descarga el raster. |
| `reduceRegion` (estadísticas, percentiles, áreas) | **Servidor** | Solo viaja al cliente un diccionario pequeño. |
| `.getInfo()` | Servidor → **Cliente** | Trae el resultado a Python. Aquí solo se usa sobre **escalares, listas cortas y diccionarios** (nombres de bandas, tamaño de colección, estadísticas). **Nunca** sobre imágenes completas. |
| Validación de parámetros, fechas, bbox, lectura de GeoJSON, tablas `pandas`, leyendas | **Cliente** (local) | Python puro. |
| `Export.image.toDrive` | **Servidor** (tarea batch) | Corre en segundo plano y deja el GeoTIFF en tu Google Drive. |

<a id="4-instalacion-de-dependencias"></a>
## 4. Instalación de dependencias

En Colab, `earthengine-api` y `geemap` suelen venir preinstalados; la celda solo instala lo que falte. En local se recomienda instalar con `pip install -r requirements.txt` dentro de un entorno virtual (la celda no instala nada localmente a menos que actives `AUTO_INSTALL_LOCAL`).

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
AUTO_INSTALL_LOCAL = False  # pon True si quieres que la celda instale paquetes en tu entorno local

# módulo importable -> especificación pip
REQUIRED_PACKAGES = {
    "ee": "earthengine-api>=1.0",
    "geemap": "geemap>=0.35",
    "pandas": "pandas>=1.5",
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]

# geemap muy antiguo (sin leyendas/colorbars/inspector actuales) -> actualizar
if importlib.util.find_spec("geemap") is not None:
    from importlib.metadata import version

    _major, _minor = (int(x) for x in version("geemap").split(".")[:2])
    if (_major, _minor) < (0, 35):
        missing.append(REQUIRED_PACKAGES["geemap"])

if not missing:
    print("✔ Todas las dependencias están instaladas.")
elif IN_COLAB or AUTO_INSTALL_LOCAL:
    print("Instalando:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✔ Instalación completa.")
else:
    print("⚠ Faltan paquetes:", ", ".join(missing))
    print("  Instálalos con:  pip install -r requirements.txt")

if IN_COLAB:
    # geemap usa ipyleaflet; Colab necesita el gestor de widgets personalizado.
    from google.colab import output
    output.enable_custom_widget_manager()

<a id="5-configuracion-global"></a>
## 5. Configuración global (parámetros editables)

**Esta es la única celda que normalmente necesitas modificar.** Todos los ejemplos usan el mismo AOI y el mismo período, para que sean comparables.

### `PROJECT_ID`
El ID de tu proyecto de Google Cloud (no el nombre ni el número). El ID **no es un secreto**, pero si prefieres no escribirlo en el notebook puedes:
- En **Colab**: guardarlo en *Secrets* (icono 🔑 de la barra lateral) con el nombre `EE_PROJECT_ID`.
- En **local**: definir la variable de entorno `EE_PROJECT_ID`.

### Área de estudio (AOI)
Por defecto: un rectángulo pequeño en el **Valle del Cauca (Colombia)** que incluye la ciudad de Cali, el valle del río Cauca y los Farallones de Cali. Cubre un gradiente altitudinal y de coberturas amplio en ~45 × 40 km, ideal para pruebas rápidas.

`AOI_METHOD` puede ser:
- `"bbox"` → `AOI_BBOX = (xmin, ymin, xmax, ymax)` en grados (WGS84, EPSG:4326).
- `"polygon"` → `AOI_POLYGON` con vértices `[lon, lat]`.
- `"geojson"` → `AOI_GEOJSON` con una ruta local o una URL (p. ej. *raw* de GitHub).
- `"admin"` → división administrativa de **FAO GAUL 2025** (`AOI_ADMIN`).

### Fechas
`START_DATE` es **inclusiva** y `END_DATE` es **exclusiva** (así funciona `filterDate` en GEE). Para todo 2024: `"2024-01-01"` a `"2025-01-01"`.

In [ ]:
import os

# ---------------------------------------------------------------------------
# 1) Proyecto de Google Cloud
# ---------------------------------------------------------------------------
PROJECT_ID = "tu-proyecto-gcp"  # <-- CAMBIA ESTO por el ID de tu proyecto

# Alternativas sin escribir el ID en el notebook (tienen prioridad si existen):
if os.environ.get("EE_PROJECT_ID"):
    PROJECT_ID = os.environ["EE_PROJECT_ID"]
try:
    from google.colab import userdata  # type: ignore

    _secret = userdata.get("EE_PROJECT_ID")
    if _secret:
        PROJECT_ID = _secret
except Exception:
    pass  # no estamos en Colab o el secreto no existe / no tiene acceso

# ---------------------------------------------------------------------------
# 2) Área de estudio (se define UNA sola vez y la usan todos los ejemplos)
# ---------------------------------------------------------------------------
AOI_METHOD = "bbox"  # "bbox" | "polygon" | "geojson" | "admin"
AOI_NAME = "Cali y Farallones (Valle del Cauca, Colombia)"

AOI_BBOX = (-76.75, 3.25, -76.35, 3.60)  # (xmin, ymin, xmax, ymax) en grados

AOI_POLYGON = [  # [lon, lat]; el último vértice puede repetir el primero
    [-76.62, 3.33],
    [-76.45, 3.33],
    [-76.45, 3.50],
    [-76.55, 3.52],
    [-76.65, 3.45],
    [-76.62, 3.33],
]

# Ruta local, ruta relativa al repositorio o URL raw de GitHub, p. ej.:
# "https://raw.githubusercontent.com/USUARIO/REPOSITORIO/main/data/aoi_example.geojson"
AOI_GEOJSON = "data/aoi_example.geojson"

AOI_ADMIN = {
    "level": 1,                  # 1 = departamento, 2 = municipio
    "country_iso3": "COL",
    "level1_name": "Valle del Cauca",
    "level2_name": None,         # p. ej. el nombre del municipio si level = 2
}

# ---------------------------------------------------------------------------
# 3) Período para datasets temporales (END_DATE es exclusiva)
# ---------------------------------------------------------------------------
START_DATE = "2024-01-01"
END_DATE = "2025-01-01"

# ---------------------------------------------------------------------------
# 4) Mapa
# ---------------------------------------------------------------------------
MAP_HEIGHT = "600px"
BASEMAP = "HYBRID"  # "HYBRID", "SATELLITE", "ROADMAP", "TERRAIN", "OpenTopoMap", ...

# ---------------------------------------------------------------------------
# 5) Exportación (desactivada por defecto para no lanzar tareas por accidente)
# ---------------------------------------------------------------------------
RUN_EXPORTS = False
EXPORT_FOLDER = "GEE_exports"  # carpeta en tu Google Drive

print(f"Proyecto: {PROJECT_ID} | AOI: {AOI_METHOD} ({AOI_NAME}) | Período: {START_DATE} → {END_DATE} (excl.)")

<a id="6-conexion-con-earth-engine"></a>
## 6. Conexión con Earth Engine

`initialize_ee()` intenta inicializar con credenciales existentes. Si no las hay, llama a `ee.Authenticate()`, que elige automáticamente el modo adecuado:

| Entorno | Modo que usa `ee.Authenticate()` | Qué pasa |
|---|---|---|
| Google Colab | `colab` | Ventana emergente de Google para autorizar tu cuenta. |
| Jupyter local con navegador | `localhost` (si no hay `gcloud`) | Se abre el navegador local. |
| Jupyter fuera de Colab / remoto | `notebook` | Enlace al *Notebook Authenticator*; requiere el rol `roles/oauthconfig.editor` en el proyecto de autenticación. |
| Con Google Cloud CLI instalado | `gcloud` (opcional) | `ee.Authenticate(auth_mode="gcloud")`. |

Las credenciales quedan en tu máquina (`~/.config/earthengine/credentials`) o en la sesión de Colab, **nunca en el repositorio**. Si fallan, `explain_ee_error()` traduce el error a una causa y solución probables.

In [ ]:
import datetime as dt
import json
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Optional, Sequence, Union

import ee
import pandas as pd

try:
    import geemap
except ImportError as exc:  # pragma: no cover
    raise ImportError("No se encontró geemap. Ejecuta la celda de instalación.") from exc

from IPython.display import display

print(f"earthengine-api {ee.__version__} | geemap {geemap.__version__} | pandas {pd.__version__}")

In [ ]:
def explain_ee_error(error: Exception, project_id: Optional[str] = None) -> str:
    """Traduce errores frecuentes de Earth Engine a un diagnóstico accionable (en español).

    Args:
        error: Excepción original.
        project_id: ID del proyecto usado, para personalizar el mensaje.

    Returns:
        Texto con la causa probable y la solución sugerida.
    """
    msg = str(error)
    low = msg.lower()
    proj = project_id or "<PROJECT_ID>"
    if any(k in low for k in ("not registered", "not signed up", "register", "not been registered")):
        hint = (f"El proyecto '{proj}' no está registrado para Earth Engine. Regístralo en "
                "https://console.cloud.google.com/earth-engine (ver GOOGLE_CLOUD_SETUP.md, paso 6).")
    elif any(k in low for k in ("has not been used in project", "service_disabled", "it is disabled", "api has not been")):
        hint = (f"La Earth Engine API no está habilitada en '{proj}'. Habilítala en "
                "https://console.cloud.google.com/apis/library/earthengine.googleapis.com")
    elif any(k in low for k in ("serviceusage", "user_project_denied", "permission to use project",
                                "caller does not have required permission")):
        hint = (f"Tu cuenta no puede usar '{proj}' como proyecto activo. Pide el rol "
                "'Service Usage Consumer' (roles/serviceusage.serviceUsageConsumer) y "
                "'Earth Engine Resource Viewer' (roles/earthengine.viewer).")
    elif "permission" in low or "403" in low:
        hint = ("Permisos insuficientes. Verifica los roles IAM de Earth Engine en el proyecto "
                "o los permisos del asset que intentas leer.")
    elif any(k in low for k in ("quota", "too many", "429", "rate limit", "resource exhausted")):
        hint = ("Se alcanzó una cuota o límite de concurrencia. Espera unos minutos, reduce el AOI/período "
                "o aumenta 'scale'. Revisa el nivel de cuota no comercial de tu proyecto.")
    elif any(k in low for k in ("not found", "does not exist", "cannot find")):
        hint = "El asset no existe o no tienes acceso. Revisa el ID exacto en el catálogo de GEE."
    elif any(k in low for k in ("authenticate", "credentials", "authorize", "not initialized", "invalid_grant")):
        hint = "Problema de autenticación. Ejecuta initialize_ee(PROJECT_ID, force_auth=True)."
    elif "user memory limit" in low or "computation timed out" in low or "too many pixels" in low:
        hint = ("La operación es demasiado pesada para una consulta interactiva. Reduce el AOI, aumenta "
                "'scale', usa tileScale mayor o lanza una exportación batch.")
    else:
        hint = "Error no catalogado. Revisa el mensaje original y la guía de depuración de GEE."
    return f"{hint}\n[Mensaje original] {msg}"


def initialize_ee(project_id: str, force_auth: bool = False, auth_mode: Optional[str] = None) -> None:
    """Autentica (si hace falta) e inicializa Earth Engine con un proyecto de Cloud.

    Args:
        project_id: ID del proyecto de Google Cloud registrado para Earth Engine.
        force_auth: Fuerza un nuevo flujo de autenticación (credenciales vencidas o de otra cuenta).
        auth_mode: Modo explícito ('colab', 'notebook', 'localhost', 'gcloud'). None = automático.

    Raises:
        ValueError: Si no se configuró PROJECT_ID.
        RuntimeError: Con un diagnóstico legible si la autenticación o inicialización fallan.
    """
    if not project_id or project_id == "tu-proyecto-gcp":
        raise ValueError("Configura PROJECT_ID en la sección 5 con el ID de tu proyecto de Google Cloud.")

    auth_keywords = ("authenticate", "credentials", "authorize", "not initialized", "invalid_grant", "refresh")
    try:
        if force_auth:
            ee.Authenticate(force=True, auth_mode=auth_mode)
        ee.Initialize(project=project_id)
    except Exception as first_error:  # noqa: BLE001
        if not any(k in str(first_error).lower() for k in auth_keywords):
            raise RuntimeError(explain_ee_error(first_error, project_id)) from first_error
        print("No hay credenciales válidas: iniciando autenticación…")
        try:
            ee.Authenticate(auth_mode=auth_mode)
            ee.Initialize(project=project_id)
        except Exception as error:  # noqa: BLE001
            raise RuntimeError(explain_ee_error(error, project_id)) from error
    print(f"✔ Earth Engine inicializado con el proyecto '{project_id}'.")


def verify_connection() -> dict:
    """Ejecuta consultas mínimas (escalares) para comprobar que la conexión funciona."""
    try:
        result = {
            "suma_servidor": ee.Number(1).add(1).getInfo(),          # cómputo trivial en el servidor
            "bandas_SRTM": ee.Image("USGS/SRTMGL1_003").bandNames().getInfo(),  # lectura de un asset público
        }
    except Exception as error:  # noqa: BLE001
        raise RuntimeError(explain_ee_error(error)) from error
    print("✔ Conexión verificada:", result)
    return result

In [ ]:
initialize_ee(PROJECT_ID)
_ = verify_connection()

<a id="7-modulos-del-codigo"></a>
## 7. Módulos del código

Las celdas de esta sección **solo definen funciones** (no consultan GEE al ejecutarse). Resumen:

| Función | Capa | Qué hace |
|---|---|---|
| `build_aoi()`, `aoi_from_bbox()`, `aoi_from_polygon()`, `aoi_from_geojson()`, `aoi_from_admin()` | Área de estudio | Convierte la configuración en un `ee.Geometry`. |
| `detect_dataset_type()`, `load_dataset()`, `validate_band()` | Carga | Detecta `Image` vs `ImageCollection`, valida existencia y bandas. |
| `filter_collection()`, `reduce_collection()`, `apply_unit_conversion()`, `prepare_image()` | Procesamiento | Filtra fechas/espacio, agrega en el tiempo, convierte unidades, recorta. |
| `get_native_scale()`, `compute_statistics()`, `compute_percentile_range()`, `compute_class_areas()` | Procesamiento | Estadísticas con `reduceRegion` (servidor). |
| `create_map()`, `add_aoi_outline()`, `add_result_to_map()`, `categorical_vis()` | Visualización | Mapas `geemap`, leyendas, barras de color. |
| `visualize_gee_dataset()` | **Función principal** | Orquesta todo lo anterior y devuelve un `GEEResult`. |
| `visualize_from_registry()` | Atajo | Usa la ficha de un dataset del `DATASET_REGISTRY`. |
| `export_image_to_drive()`, `download_small_geotiff()` | Exportación | Exportación batch a Drive o descarga directa de recortes pequeños. |

### 7.1 Utilidades, constantes y excepciones

In [ ]:
class GEEVisualizationError(Exception):
    """Error base del notebook."""


class GEEDatasetNotFoundError(GEEVisualizationError):
    """El dataset no existe o no es accesible."""


class GEEBandError(GEEVisualizationError):
    """La banda solicitada no existe en el dataset."""


class GEEEmptyCollectionError(GEEVisualizationError):
    """La colección filtrada no tiene imágenes (período o región sin datos)."""


class GEEConfigError(GEEVisualizationError):
    """Parámetros de configuración inválidos."""


# Agregaciones temporales soportadas para ImageCollection (todas se ejecutan en el servidor).
TEMPORAL_REDUCERS: dict[str, Callable[[ee.ImageCollection], ee.Image]] = {
    "mean": lambda col: col.mean(),
    "median": lambda col: col.median(),
    "sum": lambda col: col.sum(),
    "min": lambda col: col.min(),
    "max": lambda col: col.max(),
    "mode": lambda col: col.mode(),
    "mosaic": lambda col: col.mosaic(),          # útil para colecciones de teselas (p. ej. coberturas)
    "first": lambda col: ee.Image(col.first()),
}

# Conversiones de unidades con nombre (operaciones lineales sobre el servidor).
UNIT_CONVERSIONS: dict[str, Callable[[ee.Image], ee.Image]] = {
    "kelvin_to_celsius": lambda img: img.subtract(273.15),
    "m_to_mm": lambda img: img.multiply(1000),
    "mm_to_m": lambda img: img.divide(1000),
}

DEFAULT_MAX_PIXELS = 1e10


def normalize_dataset_type(dataset_type: Optional[str]) -> str:
    """Normaliza el tipo de recurso a 'Image', 'ImageCollection' o 'auto'."""
    if dataset_type is None:
        return "auto"
    key = str(dataset_type).replace("ee.", "").replace("_", "").replace(" ", "").lower()
    mapping = {"auto": "auto", "image": "Image", "imagecollection": "ImageCollection", "collection": "ImageCollection"}
    if key not in mapping:
        raise GEEConfigError(f"dataset_type='{dataset_type}' no válido. Usa 'Image', 'ImageCollection' o 'auto'.")
    return mapping[key]


def validate_dates(start_date: Optional[str], end_date: Optional[str]) -> tuple[Optional[str], Optional[str]]:
    """Valida fechas ISO (YYYY-MM-DD) del lado del cliente. END_DATE es exclusiva."""
    if start_date is None and end_date is None:
        return None, None
    if start_date is None or end_date is None:
        raise GEEConfigError("Debes indicar start_date y end_date juntos (o ninguno).")
    try:
        start = dt.date.fromisoformat(str(start_date))
        end = dt.date.fromisoformat(str(end_date))
    except ValueError as exc:
        raise GEEConfigError(f"Fechas inválidas ({start_date}, {end_date}). Usa el formato YYYY-MM-DD.") from exc
    if end <= start:
        raise GEEConfigError(f"end_date ({end}) debe ser posterior a start_date ({start}); end_date es exclusiva.")
    return start.isoformat(), end.isoformat()


def n_days(start_date: str, end_date: str) -> int:
    """Número de días del intervalo [start, end)."""
    return (dt.date.fromisoformat(end_date) - dt.date.fromisoformat(start_date)).days


def sanitize_task_name(name: str, max_len: int = 100) -> str:
    """Nombre válido para tareas de exportación (letras, números, '-', '_')."""
    clean = re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_")
    return (clean or "gee_export")[:max_len]


def categorical_palette(class_values: Sequence[int], class_colors: Sequence[str]) -> dict:
    """Construye vis_params para datos categóricos SIN reasignar los códigos originales.

    Earth Engine asigna colores de la paleta de forma lineal entre min y max. Si la paleta tiene
    exactamente (max - min + 1) colores, cada valor entero cae en su propio color. Así el
    inspector del mapa sigue mostrando el código original (p. ej. 10, 20, …, 100).
    """
    if len(class_values) != len(class_colors):
        raise GEEConfigError("class_values y class_colors deben tener la misma longitud.")
    vmin, vmax = int(min(class_values)), int(max(class_values))
    lookup = {int(v): c.lstrip("#") for v, c in zip(class_values, class_colors)}
    palette = [lookup.get(v, "000000") for v in range(vmin, vmax + 1)]
    return {"min": vmin, "max": vmax, "palette": palette}


def categorical_vis(class_info: dict) -> tuple[dict, dict]:
    """A partir de {valor: (nombre, color)} devuelve (vis_params, legend_dict)."""
    values = list(class_info.keys())
    colors = [class_info[v][1] for v in values]
    vis = categorical_palette(values, colors)
    legend = {f"{v} – {class_info[v][0]}": "#" + class_info[v][1].lstrip("#") for v in values}
    return vis, legend

### 7.2 Área de estudio (AOI)

Todas las funciones devuelven un `ee.Geometry` en EPSG:4326. La validación de coordenadas se hace localmente antes de enviar nada al servidor.

In [ ]:
SAMPLE_GEOJSON = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "AOI de ejemplo - Cali (Valle del Cauca, Colombia)"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [-76.62, 3.33], [-76.45, 3.33], [-76.45, 3.50],
                    [-76.55, 3.52], [-76.65, 3.45], [-76.62, 3.33],
                ]],
            },
        }
    ],
}


def validate_bbox(bbox: Sequence[float]) -> tuple[float, float, float, float]:
    """Valida (xmin, ymin, xmax, ymax) en grados geográficos."""
    if len(bbox) != 4:
        raise GEEConfigError("El bbox debe tener 4 valores: (xmin, ymin, xmax, ymax).")
    xmin, ymin, xmax, ymax = map(float, bbox)
    if not (-180 <= xmin < xmax <= 180):
        raise GEEConfigError(f"Longitudes inválidas: xmin={xmin}, xmax={xmax} (se requiere -180 ≤ xmin < xmax ≤ 180).")
    if not (-90 <= ymin < ymax <= 90):
        raise GEEConfigError(f"Latitudes inválidas: ymin={ymin}, ymax={ymax} (se requiere -90 ≤ ymin < ymax ≤ 90).")
    return xmin, ymin, xmax, ymax


def validate_polygon(coords: Sequence[Sequence[float]]) -> list[list[float]]:
    """Valida una lista de vértices [lon, lat] y cierra el anillo si hace falta."""
    ring = [[float(x), float(y)] for x, y in coords]
    if len(ring) < 3:
        raise GEEConfigError("Un polígono necesita al menos 3 vértices.")
    for lon, lat in ring:
        if not (-180 <= lon <= 180 and -90 <= lat <= 90):
            raise GEEConfigError(f"Vértice fuera de rango: [{lon}, {lat}]. ¿Invertiste lon/lat?")
    if ring[0] != ring[-1]:
        ring.append(ring[0])
    return ring


def load_geojson_dict(source: Union[str, Path, dict]) -> dict:
    """Lee un GeoJSON desde dict, archivo local o URL http(s). Si la ruta local no existe,
    busca en ubicaciones típicas del repositorio y, como último recurso, escribe el GeoJSON de ejemplo."""
    if isinstance(source, dict):
        return source
    src = str(source)
    if src.startswith(("http://", "https://")):
        import urllib.request

        with urllib.request.urlopen(src, timeout=30) as resp:  # noqa: S310 (URL definida por el usuario)
            return json.loads(resp.read().decode("utf-8"))
    candidates = [Path(src), Path("..") / src, Path.cwd() / src]
    for path in candidates:
        if path.is_file():
            return json.loads(path.read_text(encoding="utf-8"))
    # Colab abre solo el notebook (no el repositorio): generamos el archivo de ejemplo.
    fallback = Path(Path(src).name)
    fallback.write_text(json.dumps(SAMPLE_GEOJSON, indent=2), encoding="utf-8")
    warnings.warn(f"No se encontró '{src}'. Se escribió el GeoJSON de ejemplo en '{fallback.resolve()}'.")
    return SAMPLE_GEOJSON


def geojson_geometries(gj: dict) -> list[dict]:
    """Extrae las geometrías de un GeoJSON (FeatureCollection, Feature o Geometry)."""
    gtype = gj.get("type")
    if gtype == "FeatureCollection":
        geoms = [f["geometry"] for f in gj.get("features", []) if f.get("geometry")]
    elif gtype == "Feature":
        geoms = [gj["geometry"]]
    elif gtype in {"Polygon", "MultiPolygon", "Point", "MultiPoint", "LineString", "MultiLineString", "GeometryCollection"}:
        geoms = [gj]
    else:
        raise GEEConfigError(f"Tipo de GeoJSON no soportado: {gtype}")
    if not geoms:
        raise GEEConfigError("El GeoJSON no contiene geometrías.")
    return geoms


def aoi_from_bbox(bbox: Sequence[float]) -> ee.Geometry:
    """AOI rectangular a partir de (xmin, ymin, xmax, ymax)."""
    return ee.Geometry.Rectangle(list(validate_bbox(bbox)), proj="EPSG:4326", geodesic=False)


def aoi_from_polygon(coords: Sequence[Sequence[float]]) -> ee.Geometry:
    """AOI poligonal a partir de vértices [lon, lat]."""
    return ee.Geometry.Polygon([validate_polygon(coords)], proj="EPSG:4326", geodesic=False)


def aoi_from_geojson(source: Union[str, Path, dict]) -> ee.Geometry:
    """AOI a partir de un GeoJSON (se unen todas las geometrías)."""
    geoms = geojson_geometries(load_geojson_dict(source))
    features = [ee.Feature(ee.Geometry(g)) for g in geoms]
    return ee.FeatureCollection(features).geometry(maxError=1)


GAUL_LEVELS = {1: "FAO/GAUL/2025/level1", 2: "FAO/GAUL/2025/level2"}


def list_admin_names(level: int, country_iso3: str, level1_name: Optional[str] = None) -> list[str]:
    """Lista (getInfo de una lista corta) los nombres disponibles en FAO GAUL 2025 para ayudar a escribirlos bien."""
    fc = ee.FeatureCollection(GAUL_LEVELS[level]).filter(ee.Filter.eq("ISO3_CODE", country_iso3))
    if level == 2 and level1_name:
        fc = fc.filter(ee.Filter.eq("GAUL1_NAME", level1_name))
    return sorted(fc.aggregate_array(f"GAUL{level}_NAME").distinct().getInfo())


def aoi_from_admin(level: int, country_iso3: str, level1_name: str, level2_name: Optional[str] = None) -> ee.Geometry:
    """AOI desde una división administrativa de FAO GAUL 2025 (nivel 1 o 2).

    Esquema de la tabla verificado en el catálogo: ISO3_CODE, GAUL0_NAME, GAUL1_NAME, GAUL2_NAME.
    """
    if level not in GAUL_LEVELS:
        raise GEEConfigError("level debe ser 1 (departamento) o 2 (municipio).")
    fc = ee.FeatureCollection(GAUL_LEVELS[level]).filter(ee.Filter.eq("ISO3_CODE", country_iso3))
    fc = fc.filter(ee.Filter.eq("GAUL1_NAME", level1_name))
    if level == 2:
        if not level2_name:
            raise GEEConfigError("Para level=2 debes indicar level2_name.")
        fc = fc.filter(ee.Filter.eq("GAUL2_NAME", level2_name))
    if fc.size().getInfo() == 0:
        parent = level1_name if level == 2 else None
        options = list_admin_names(level, country_iso3, parent)
        raise GEEConfigError(
            f"No se encontró la unidad administrativa. Nombres disponibles (nivel {level}):\n{options}"
        )
    return fc.geometry(maxError=100)


def build_aoi(method: str = None) -> ee.Geometry:
    """Construye el AOI global a partir de la configuración de la sección 5."""
    method = (method or AOI_METHOD).lower()
    if method == "bbox":
        return aoi_from_bbox(AOI_BBOX)
    if method == "polygon":
        return aoi_from_polygon(AOI_POLYGON)
    if method == "geojson":
        return aoi_from_geojson(AOI_GEOJSON)
    if method == "admin":
        return aoi_from_admin(**AOI_ADMIN)
    raise GEEConfigError(f"AOI_METHOD='{method}' no válido. Usa 'bbox', 'polygon', 'geojson' o 'admin'.")


def to_geometry(aoi: Any) -> ee.Geometry:
    """Acepta ee.Geometry/Feature/FeatureCollection, bbox (4 números), lista de vértices o GeoJSON dict."""
    if isinstance(aoi, ee.Geometry):
        return aoi
    if isinstance(aoi, (ee.Feature, ee.FeatureCollection)):
        return aoi.geometry()
    if isinstance(aoi, dict):
        return aoi_from_geojson(aoi)
    if isinstance(aoi, (list, tuple)):
        if len(aoi) == 4 and all(isinstance(v, (int, float)) for v in aoi):
            return aoi_from_bbox(aoi)
        return aoi_from_polygon(aoi)
    raise GEEConfigError(f"No sé convertir un objeto {type(aoi).__name__} en AOI.")

### 7.3 Carga de datos

`detect_dataset_type()` consulta los **metadatos** del asset (`ee.data.getAsset`), no sus píxeles, para saber si es `Image` o `ImageCollection`. La validación de la banda trae solo la lista de nombres de bandas.

In [ ]:
def detect_dataset_type(dataset_id: str) -> str:
    """Devuelve 'Image' o 'ImageCollection' leyendo los metadatos del asset."""
    try:
        asset = ee.data.getAsset(dataset_id)
    except Exception as error:  # noqa: BLE001
        raise GEEDatasetNotFoundError(
            f"No se pudo acceder al dataset '{dataset_id}'. Verifica el ID exacto en "
            f"https://developers.google.com/earth-engine/datasets\n{explain_ee_error(error)}"
        ) from error
    asset_type = str(asset.get("type", "")).upper()
    mapping = {"IMAGE": "Image", "IMAGE_COLLECTION": "ImageCollection"}
    if asset_type not in mapping:
        raise GEEConfigError(f"'{dataset_id}' es de tipo {asset_type}; este flujo admite Image o ImageCollection.")
    return mapping[asset_type]


def load_dataset(dataset_id: str, dataset_type: str = "auto") -> tuple[Union[ee.Image, ee.ImageCollection], str]:
    """Carga un dataset del catálogo como ee.Image o ee.ImageCollection.

    Returns:
        (objeto_ee, tipo) con tipo en {'Image', 'ImageCollection'}.
    """
    dtype = normalize_dataset_type(dataset_type)
    if dtype == "auto":
        dtype = detect_dataset_type(dataset_id)
    obj = ee.Image(dataset_id) if dtype == "Image" else ee.ImageCollection(dataset_id)
    return obj, dtype


def validate_band(image: ee.Image, band: str, dataset_id: str = "") -> list[str]:
    """Comprueba que `band` exista. Trae al cliente solo la lista de nombres de bandas."""
    try:
        available = ee.Image(image).bandNames().getInfo()
    except Exception as error:  # noqa: BLE001
        raise GEEDatasetNotFoundError(
            f"No se pudieron leer las bandas de '{dataset_id}'.\n{explain_ee_error(error)}"
        ) from error
    if band not in available:
        raise GEEBandError(f"La banda '{band}' no existe en '{dataset_id}'. Bandas disponibles: {available}")
    return available

### 7.4 Procesamiento

Flujo de `prepare_image()`:

1. **Image** → selecciona la banda (las fechas se ignoran con un aviso).
2. **ImageCollection** → valida la banda en la primera imagen de la colección completa → filtra por AOI (`filterBounds`) y fechas (`filterDate`) → verifica que no quede vacía → **agrega en el tiempo** (`mean`, `sum`, `median`…) → restaura la proyección nativa (las agregaciones de colecciones pierden la proyección por defecto).
3. Convierte unidades (después de la agregación; ver nota) y, opcionalmente, recorta al AOI.

> **Nota metodológica sobre conversiones con desplazamiento (K → °C).** Para la **media**, convertir antes o después de agregar es equivalente (la media es lineal). Para la **suma** no: sumar N imágenes en °C equivale a restar N × 273,15. Por eso el código advierte si combinas `sum` con `kelvin_to_celsius`.

In [ ]:
def get_native_scale(image: ee.Image) -> float:
    """Escala nominal (m) de la primera banda: un escalar pequeño vía getInfo()."""
    return float(ee.Image(image).projection().nominalScale().getInfo())


def filter_collection(collection: ee.ImageCollection, aoi: ee.Geometry,
                      start_date: Optional[str], end_date: Optional[str], dataset_id: str = "") -> tuple[ee.ImageCollection, int]:
    """Filtra por AOI y fechas y verifica que la colección no quede vacía."""
    col = collection.filterBounds(aoi)
    if start_date and end_date:
        col = col.filterDate(start_date, end_date)
    size = col.size().getInfo()
    if size == 0:
        # Solo en caso de error consultamos el rango temporal del dataset (metadatos agregados).
        try:
            full = collection.filterBounds(aoi)
            t0 = ee.Date(full.aggregate_min("system:time_start")).format("YYYY-MM-dd").getInfo()
            t1 = ee.Date(full.aggregate_max("system:time_start")).format("YYYY-MM-dd").getInfo()
            rng = f" Datos disponibles en el AOI: {t0} → {t1}."
        except Exception:  # noqa: BLE001
            rng = ""
        raise GEEEmptyCollectionError(
            f"La colección '{dataset_id}' no tiene imágenes para el AOI y el período "
            f"[{start_date}, {end_date}).{rng}"
        )
    return col, size


def reduce_collection(collection: ee.ImageCollection, reducer: str) -> ee.Image:
    """Agregación temporal con nombre; conserva la proyección nativa como proyección por defecto."""
    if reducer not in TEMPORAL_REDUCERS:
        raise GEEConfigError(f"reducer='{reducer}' no soportado. Opciones: {list(TEMPORAL_REDUCERS)}")
    native_proj = ee.Image(collection.first()).projection()
    reduced = TEMPORAL_REDUCERS[reducer](collection)
    if reducer not in ("first",):
        reduced = reduced.setDefaultProjection(native_proj)
    return reduced


def apply_unit_conversion(image: ee.Image, conversion: Union[None, str, Callable[[ee.Image], ee.Image]]) -> ee.Image:
    """Aplica una conversión de unidades con nombre o una función personalizada ee.Image -> ee.Image."""
    if conversion is None:
        return image
    if callable(conversion):
        return conversion(image)
    if conversion not in UNIT_CONVERSIONS:
        raise GEEConfigError(f"Conversión '{conversion}' no reconocida. Opciones: {list(UNIT_CONVERSIONS)} o una función.")
    return UNIT_CONVERSIONS[conversion](image)


def prepare_image(dataset_id: str, band: str, aoi: ee.Geometry, dataset_type: str = "auto",
                  start_date: Optional[str] = None, end_date: Optional[str] = None,
                  reducer: str = "mean", unit_conversion: Any = None, clip: bool = True) -> tuple[ee.Image, dict]:
    """Carga + filtra + agrega + convierte + recorta. Devuelve la imagen procesada y metadatos."""
    start_date, end_date = validate_dates(start_date, end_date)
    obj, dtype = load_dataset(dataset_id, dataset_type)
    meta: dict[str, Any] = {"dataset_id": dataset_id, "dataset_type": dtype, "band": band}

    if dtype == "Image":
        validate_band(obj, band, dataset_id)
        if start_date:
            warnings.warn(f"'{dataset_id}' es una ee.Image: start_date/end_date se ignoran.")
        image = obj.select(band)
        meta["native_scale_source"] = image
    else:
        first_full = ee.Image(obj.limit(1).first())
        validate_band(first_full, band, dataset_id)
        col, size = filter_collection(obj.select(band), aoi, start_date, end_date, dataset_id)
        meta.update({"n_images": size, "start_date": start_date, "end_date": end_date, "reducer": reducer})
        if reducer == "sum" and unit_conversion == "kelvin_to_celsius":
            warnings.warn("Sumar temperaturas y luego restar 273.15 no es físicamente correcto. Usa 'mean'.")
        image = reduce_collection(col, reducer)
        meta["native_scale_source"] = ee.Image(col.first())

    image = apply_unit_conversion(image, unit_conversion)
    if clip:
        image = image.clip(aoi)
    return image, meta


def compute_statistics(image: ee.Image, band: str, aoi: ee.Geometry, scale: float,
                       max_pixels: float = DEFAULT_MAX_PIXELS, tile_scale: int = 4) -> pd.DataFrame:
    """Mínimo, máximo, media, desviación estándar y conteo de píxeles en el AOI (cómputo en servidor)."""
    reducer = (ee.Reducer.mean()
               .combine(ee.Reducer.minMax(), sharedInputs=True)
               .combine(ee.Reducer.stdDev(), sharedInputs=True)
               .combine(ee.Reducer.count(), sharedInputs=True))
    stats = image.select(band).reduceRegion(
        reducer=reducer, geometry=aoi, scale=scale, maxPixels=max_pixels, bestEffort=True, tileScale=tile_scale
    ).getInfo()
    clean = {k.replace(f"{band}_", ""): v for k, v in stats.items()}
    order = ["min", "max", "mean", "stdDev", "count"]
    row = {k: clean.get(k) for k in order}
    return pd.DataFrame([row], index=[band]).rename(columns={
        "min": "mínimo", "max": "máximo", "mean": "media", "stdDev": "desv. estándar", "count": "n píxeles"
    })


def compute_percentile_range(image: ee.Image, band: str, aoi: ee.Geometry, scale: float,
                             low: int = 2, high: int = 98) -> tuple[float, float]:
    """Rango robusto (percentiles) para estirar la paleta sin que valores extremos la dominen."""
    res = image.select(band).reduceRegion(
        reducer=ee.Reducer.percentile([low, high]), geometry=aoi, scale=scale,
        maxPixels=DEFAULT_MAX_PIXELS, bestEffort=True, tileScale=4,
    ).getInfo()
    vmin, vmax = res.get(f"{band}_p{low}"), res.get(f"{band}_p{high}")
    if vmin is None or vmax is None:
        raise GEEEmptyCollectionError("No hay píxeles válidos en el AOI para calcular el rango de visualización.")
    if vmin == vmax:
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)


def compute_class_areas(image: ee.Image, band: str, aoi: ee.Geometry, scale: float,
                        class_info: Optional[dict] = None) -> pd.DataFrame:
    """Área (ha) y porcentaje por clase de un raster categórico, usando ee.Image.pixelArea()."""
    area_img = ee.Image.pixelArea().divide(1e4).addBands(image.select(band))
    groups = area_img.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="clase"),
        geometry=aoi, scale=scale, maxPixels=DEFAULT_MAX_PIXELS, tileScale=4,
    ).get("groups").getInfo()
    df = pd.DataFrame(groups).rename(columns={"sum": "area_ha"})
    if df.empty:
        return df
    df["clase"] = df["clase"].astype(int)
    if class_info:
        df["nombre"] = df["clase"].map(lambda c: class_info.get(c, ("(sin nombre)",))[0])
    df["porcentaje"] = 100 * df["area_ha"] / df["area_ha"].sum()
    cols = ["clase", "nombre", "area_ha", "porcentaje"] if class_info else ["clase", "area_ha", "porcentaje"]
    return df[cols].sort_values("area_ha", ascending=False).reset_index(drop=True).round(2)

### 7.5 Visualización

- `create_map()` crea un mapa `geemap` con mapa base y control de capas (activar/desactivar).
- `add_aoi_outline()` dibuja solo el borde del AOI (pintado en el servidor como una imagen de 1 banda).
- `add_result_to_map()` agrega un resultado con su leyenda (categórica) o barra de colores (continua).
- El mapa de `geemap` incluye zoom, desplazamiento, pantalla completa y un **inspector** para consultar valores de píxel.

In [ ]:
def create_map(center: Optional[Sequence[float]] = None, zoom: Optional[int] = None,
               height: str = None, basemap: str = None) -> "geemap.Map":
    """Mapa interactivo con control de capas. center = (lon, lat)."""
    m = geemap.Map(height=height or MAP_HEIGHT, basemap=basemap or BASEMAP)
    if center is not None:
        m.set_center(center[0], center[1], zoom or 10)
    m.add_layer_control()
    return m


def add_aoi_outline(m: "geemap.Map", aoi: ee.Geometry, name: str = "Área de interés",
                    color: str = "FF0000", width: int = 2) -> None:
    """Dibuja solo el contorno del AOI (sin relleno)."""
    outline = ee.Image().byte().paint(featureCollection=ee.FeatureCollection([ee.Feature(aoi)]), color=1, width=width)
    m.add_layer(outline, {"palette": [color]}, name)


@dataclass
class GEEResult:
    """Resultado de visualize_gee_dataset()."""

    image: ee.Image
    vis_params: dict
    layer_name: str
    metadata: dict
    scale: float
    legend: Optional[dict] = None
    colorbar_label: Optional[str] = None
    statistics: Optional[pd.DataFrame] = None
    map: Any = None

    def summary(self) -> pd.DataFrame:
        """Tabla de metadatos legible."""
        keys = ["dataset_id", "dataset_type", "band", "start_date", "end_date", "reducer", "n_images"]
        rows = {k: self.metadata.get(k) for k in keys if self.metadata.get(k) is not None}
        rows["escala de trabajo (m)"] = round(self.scale, 1)
        return pd.DataFrame.from_dict(rows, orient="index", columns=["valor"])


def add_result_to_map(m: "geemap.Map", result: GEEResult, shown: bool = True, opacity: float = 1.0,
                      legend_position: str = "bottomright", add_guide: bool = True) -> None:
    """Agrega un GEEResult a un mapa con su leyenda o barra de colores."""
    m.add_layer(result.image, result.vis_params, result.layer_name, shown, opacity)
    if not add_guide:
        return
    if result.legend:
        m.add_legend(title=result.layer_name, legend_dict=result.legend, position=legend_position)
    elif "palette" in result.vis_params and "min" in result.vis_params:
        m.add_colorbar(result.vis_params, label=result.colorbar_label or result.layer_name,
                       layer_name=result.layer_name, position=legend_position)

### 7.6 Función principal: `visualize_gee_dataset()`

Es la función genérica del notebook. Cambiando sus argumentos visualizas **cualquier** `Image` o `ImageCollection` del catálogo.

| Parámetro | Tipo | Descripción |
|---|---|---|
| `dataset_id` | `str` | ID del asset en GEE (p. ej. `"USGS/SRTMGL1_003"`). |
| `band` | `str` | Banda a visualizar. |
| `aoi` | `ee.Geometry`, bbox, vértices o GeoJSON | Área de interés. |
| `dataset_type` | `"Image"`, `"ImageCollection"` o `"auto"` | `"auto"` lo detecta por metadatos. |
| `start_date`, `end_date` | `str` `YYYY-MM-DD` | Solo para colecciones; `end_date` exclusiva. |
| `reducer` | `str` | `mean`, `sum`, `median`, `min`, `max`, `mode`, `mosaic`, `first`. |
| `scale` | `float` (m) | Resolución para estadísticas/rangos. `None` = escala nativa. |
| `vis_params` | `dict` | `min`, `max`, `palette`. Si faltan `min`/`max` y `auto_stretch=True`, se calculan con percentiles 2–98. |
| `class_info` | `dict` | `{valor: (nombre, color)}` para datos categóricos (genera paleta y leyenda). |
| `unit_conversion` | `str` o función | `"kelvin_to_celsius"`, `"m_to_mm"`, … o una función `ee.Image → ee.Image`. |
| `clip` | `bool` | Recorta al AOI. |
| `layer_name` | `str` | Nombre de la capa en el mapa. |
| `map_center`, `zoom` | `(lon, lat)`, `int` | Opcionales; por defecto centra en el AOI. |
| `map_obj` | `geemap.Map` | Para agregar la capa a un mapa existente. |
| `compute_stats` | `bool` | Calcula mínimo, máximo, media, desviación y conteo. |
| `colorbar_label` | `str` | Etiqueta (con unidades) de la barra de colores. |
| `show_map` | `bool` | `False` para solo procesar (sin crear mapa). |

In [ ]:
def visualize_gee_dataset(
    dataset_id: str,
    band: str,
    aoi: Any,
    dataset_type: str = "auto",
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    reducer: str = "mean",
    scale: Optional[float] = None,
    vis_params: Optional[dict] = None,
    clip: bool = True,
    layer_name: Optional[str] = None,
    map_center: Optional[Sequence[float]] = None,
    zoom: Optional[int] = None,
    map_obj: Any = None,
    unit_conversion: Any = None,
    class_info: Optional[dict] = None,
    colorbar_label: Optional[str] = None,
    compute_stats: bool = False,
    auto_stretch: bool = True,
    show_aoi: bool = True,
    show_map: bool = True,
) -> GEEResult:
    """Consulta, procesa y visualiza un dataset de Google Earth Engine.

    Maneja ee.Image y ee.ImageCollection, valida dataset/banda/período y devuelve un GEEResult
    con la imagen procesada (objeto ee del lado del servidor), vis_params, estadísticas y mapa.

    Ejemplo:
        >>> res = visualize_gee_dataset("USGS/SRTMGL1_003", "elevation", AOI, compute_stats=True)
        >>> res.map  # muestra el mapa
    """
    geom = to_geometry(aoi)
    layer_name = layer_name or f"{dataset_id} · {band}"

    # 1) Carga + procesamiento (servidor)
    image, meta = prepare_image(dataset_id, band, geom, dataset_type, start_date, end_date,
                                reducer, unit_conversion, clip)

    # 2) Escala de trabajo
    native_src = meta.pop("native_scale_source")
    work_scale = float(scale) if scale else get_native_scale(native_src)

    # 3) Parámetros de visualización
    legend = None
    if class_info:
        vis, legend = categorical_vis(class_info)
        vis.update({k: v for k, v in (vis_params or {}).items() if k not in ("min", "max", "palette")})
    else:
        vis = dict(vis_params or {})
        if auto_stretch and ("min" not in vis or "max" not in vis):
            pmin, pmax = compute_percentile_range(image, band, geom, work_scale)
            vis.setdefault("min", round(pmin, 2))
            vis.setdefault("max", round(pmax, 2))
        vis.setdefault("palette", ["000000", "ffffff"])

    # 4) Estadísticas (servidor → dict pequeño)
    stats = compute_statistics(image, band, geom, work_scale) if compute_stats else None

    result = GEEResult(image=image, vis_params=vis, layer_name=layer_name, metadata=meta,
                       scale=work_scale, legend=legend, colorbar_label=colorbar_label, statistics=stats)

    # 5) Mapa (teselas renderizadas por GEE; no se descarga el raster)
    if show_map:
        m = map_obj if map_obj is not None else create_map()
        if map_obj is None and show_aoi:
            add_aoi_outline(m, geom)
        add_result_to_map(m, result)
        if map_center is not None:
            m.set_center(map_center[0], map_center[1], zoom or 10)
        elif map_obj is None:
            m.center_object(geom, zoom)
        result.map = m
    return result

### 7.7 Exportación (opcional)

- `export_image_to_drive()` lanza una **tarea batch** (servidor) que escribe un GeoTIFF en tu Google Drive. Es la vía recomendada para áreas grandes o resoluciones finas.
- `download_small_geotiff()` descarga directamente un recorte **pequeño** con `geemap.ee_export_image` (usa `getDownloadURL`, que tiene un límite de tamaño por solicitud). Si falla por tamaño, usa la exportación a Drive o aumenta `scale`.

In [ ]:
def export_image_to_drive(image: ee.Image, description: str, aoi: ee.Geometry, scale: float,
                          folder: str = None, crs: str = "EPSG:4326", start: bool = False) -> ee.batch.Task:
    """Crea (y opcionalmente inicia) una exportación GeoTIFF a Google Drive."""
    name = sanitize_task_name(description)
    task = ee.batch.Export.image.toDrive(
        image=image.toFloat(), description=name, folder=folder or EXPORT_FOLDER, fileNamePrefix=name,
        region=aoi, scale=scale, crs=crs, maxPixels=DEFAULT_MAX_PIXELS, fileFormat="GeoTIFF",
    )
    if start:
        task.start()
        print(f"▶ Tarea '{name}' iniciada. Revisa su estado con task.status() o en https://code.earthengine.google.com/tasks")
    else:
        print(f"Tarea '{name}' creada (no iniciada). Llama a task.start() para ejecutarla.")
    return task


def download_small_geotiff(image: ee.Image, filename: str, aoi: ee.Geometry, scale: float,
                           crs: str = "EPSG:4326") -> Path:
    """Descarga directa de un recorte pequeño a un GeoTIFF local."""
    out = Path(filename)
    out.parent.mkdir(parents=True, exist_ok=True)
    geemap.ee_export_image(image, filename=str(out), scale=scale, region=aoi, crs=crs, file_per_band=False)
    return out

### 7.8 Registro de datasets (`DATASET_REGISTRY`)

Cada dataset es una **ficha de configuración**: ID, tipo, banda, agregación, conversión de unidades, visualización y documentación. Para agregar un dataset nuevo **solo se agrega una ficha**; la lógica no cambia (ver sección 15).

Todos los IDs, bandas, unidades y resoluciones se verificaron contra el catálogo oficial de GEE (enlaces en cada ficha y en la sección 18).

In [ ]:
WORLDCOVER_CLASSES = {  # valor: (nombre, color) — tabla de clases de la banda 'Map' (catálogo GEE)
    10: ("Cobertura arbórea", "006400"),
    20: ("Arbustal", "ffbb22"),
    30: ("Herbazal", "ffff4c"),
    40: ("Cultivos", "f096ff"),
    50: ("Área construida", "fa0000"),
    60: ("Suelo desnudo / vegetación escasa", "b4b4b4"),
    70: ("Nieve y hielo", "f0f0f0"),
    80: ("Cuerpos de agua permanentes", "0064c8"),
    90: ("Humedal herbáceo", "0096a0"),
    95: ("Manglar", "00cf75"),
    100: ("Musgos y líquenes", "fae6a0"),
}

PALETTES = {
    "elevation": ["0b6a2c", "5aa542", "d8d26b", "b98b4e", "8c5a34", "d9d9d9", "ffffff"],
    "slope": ["ffffcc", "fed976", "fd8d3c", "e31a1c", "800026"],
    "precipitation": ["f7fbff", "c6dbef", "6baed6", "2171b5", "08519c", "08306b"],
    "temperature": ["313695", "4575b4", "74add1", "abd9e9", "e0f3f8", "ffffbf",
                    "fee090", "fdae61", "f46d43", "d73027", "a50026"],
}

DATASET_REGISTRY: dict[str, dict] = {
    "landcover_worldcover": {
        "title": "ESA WorldCover 10 m v200 (2021)",
        "dataset_id": "ESA/WorldCover/v200",
        "dataset_type": "ImageCollection",
        "band": "Map",
        "reducer": "mosaic",
        "temporal": False,
        "unit_conversion": None,
        "class_info": WORLDCOVER_CLASSES,
        "units": ("código de clase", "código de clase"),
        "resolution": "10 m; mapa anual de referencia 2021",
        "url": "https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200",
        "license": "CC-BY-4.0",
    },
    "dem_srtm": {
        "title": "NASA SRTM Digital Elevation 30 m (v3)",
        "dataset_id": "USGS/SRTMGL1_003",
        "dataset_type": "Image",
        "band": "elevation",
        "reducer": None,
        "temporal": False,
        "unit_conversion": None,
        "vis_params": {"palette": PALETTES["elevation"]},
        "colorbar_label": "Elevación (m)",
        "units": ("m", "m"),
        "resolution": "1 arco-segundo (~30 m); misión de febrero de 2000",
        "url": "https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003",
        "license": "Dominio público (NASA/USGS)",
    },
    "precip_chirps_sum": {
        "title": "CHIRPS Daily v2.0 — precipitación acumulada",
        "dataset_id": "UCSB-CHG/CHIRPS/DAILY",
        "dataset_type": "ImageCollection",
        "band": "precipitation",
        "reducer": "sum",
        "temporal": True,
        "unit_conversion": None,
        "vis_params": {"palette": PALETTES["precipitation"]},
        "colorbar_label": "Precipitación acumulada (mm)",
        "units": ("mm/d", "mm (total del período)"),
        "resolution": "0,05° (~5,5 km); diaria",
        "url": "https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY",
        "license": "Dominio público",
    },
    "precip_chirps_mean": {
        "title": "CHIRPS Daily v2.0 — precipitación media diaria",
        "dataset_id": "UCSB-CHG/CHIRPS/DAILY",
        "dataset_type": "ImageCollection",
        "band": "precipitation",
        "reducer": "mean",
        "temporal": True,
        "unit_conversion": None,
        "vis_params": {"palette": PALETTES["precipitation"]},
        "colorbar_label": "Precipitación media (mm/día)",
        "units": ("mm/d", "mm/d (promedio del período)"),
        "resolution": "0,05° (~5,5 km); diaria",
        "url": "https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY",
        "license": "Dominio público",
    },
    "temp_era5land": {
        "title": "ERA5-Land Daily Aggregates — temperatura del aire a 2 m",
        "dataset_id": "ECMWF/ERA5_LAND/DAILY_AGGR",
        "dataset_type": "ImageCollection",
        "band": "temperature_2m",
        "reducer": "mean",
        "temporal": True,
        "unit_conversion": "kelvin_to_celsius",
        "vis_params": {"palette": PALETTES["temperature"]},
        "colorbar_label": "Temperatura media a 2 m (°C)",
        "units": ("K", "°C"),
        "resolution": "0,1° (~11 km, 11 132 m en GEE); diaria (agregada desde horaria)",
        "url": "https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR",
        "license": "Copernicus (ver términos en el catálogo)",
    },
}


def visualize_from_registry(key: str, aoi: Any, start_date: Optional[str] = None,
                            end_date: Optional[str] = None, **overrides: Any) -> GEEResult:
    """Ejecuta visualize_gee_dataset() con la ficha `key` del registro. Los kwargs sobrescriben la ficha."""
    if key not in DATASET_REGISTRY:
        raise GEEConfigError(f"'{key}' no está en DATASET_REGISTRY. Opciones: {list(DATASET_REGISTRY)}")
    spec = DATASET_REGISTRY[key]
    params = {
        "dataset_id": spec["dataset_id"],
        "band": spec["band"],
        "aoi": aoi,
        "dataset_type": spec.get("dataset_type", "auto"),
        "reducer": spec.get("reducer") or "mean",
        "vis_params": spec.get("vis_params"),
        "class_info": spec.get("class_info"),
        "unit_conversion": spec.get("unit_conversion"),
        "colorbar_label": spec.get("colorbar_label"),
        "layer_name": spec["title"],
    }
    if spec.get("temporal"):
        params.update({"start_date": start_date or START_DATE, "end_date": end_date or END_DATE})
    params.update(overrides)
    return visualize_gee_dataset(**params)


def registry_table() -> pd.DataFrame:
    """Resumen del registro (fuente, unidades, resolución)."""
    rows = []
    for key, s in DATASET_REGISTRY.items():
        rows.append({"clave": key, "dataset": s["title"], "ID": s["dataset_id"], "banda": s["band"],
                     "agregación": s.get("reducer") or "—", "unidad original": s["units"][0],
                     "unidad final": s["units"][1], "resolución": s["resolution"]})
    return pd.DataFrame(rows).set_index("clave")


registry_table()

<a id="8-area-de-estudio"></a>
## 8. Definición del área de estudio (AOI)

El AOI global `AOI` se construye **una sola vez** a partir de `AOI_METHOD` (sección 5). Abajo se muestran las cuatro formas de definirlo; todas devuelven un `ee.Geometry`.

```python
aoi_bbox    = aoi_from_bbox((-76.75, 3.25, -76.35, 3.60))          # 1. bounding box
aoi_poly    = aoi_from_polygon(AOI_POLYGON)                         # 2. polígono manual
aoi_geojson = aoi_from_geojson("data/aoi_example.geojson")          # 3. archivo GeoJSON (o URL)
aoi_admin   = aoi_from_admin(level=1, country_iso3="COL",
                             level1_name="Valle del Cauca")         # 4. división administrativa (FAO GAUL 2025)
```

> Para la opción 4, si el nombre no coincide exactamente, la función muestra la lista de nombres válidos. También puedes consultarla con `list_admin_names(1, "COL")`. FAO GAUL 2025 está en el catálogo de GEE con licencia CC-BY-4.0.

In [ ]:
AOI = build_aoi()

# Área del AOI en km² (escalar pequeño)
aoi_area_km2 = AOI.area(maxError=1).divide(1e6).getInfo()
print(f"AOI '{AOI_NAME}' ({AOI_METHOD}): {aoi_area_km2:,.1f} km²")

m_aoi = create_map()
add_aoi_outline(m_aoi, AOI, name=f"AOI ({AOI_METHOD})")
# Comparación visual de los cuatro métodos (los tres primeros no requieren consultas pesadas)
add_aoi_outline(m_aoi, aoi_from_polygon(AOI_POLYGON), name="AOI polígono manual", color="00FFFF")
add_aoi_outline(m_aoi, aoi_from_geojson(AOI_GEOJSON), name="AOI GeoJSON", color="FFFF00", width=1)
m_aoi.center_object(AOI)
m_aoi

In [ ]:
# (Opcional) Ejemplo con división administrativa. Descomenta para probar:
# aoi_valle = aoi_from_admin(level=1, country_iso3="COL", level1_name="Valle del Cauca")
# list_admin_names(2, "COL", level1_name="Valle del Cauca")[:20]   # nombres de municipios disponibles

<a id="9-ejemplo-1-cobertura-del-suelo"></a>
## 9. Ejemplo 1 — Cobertura y uso del suelo (ESA WorldCover v200)

| Ítem | Detalle |
|---|---|
| **Fuente** | Agencia Espacial Europea (ESA), proyecto WorldCover. Clasificación global derivada de Sentinel-1 (radar) y Sentinel-2 (óptico). |
| **Enlace GEE** | [ESA/WorldCover/v200](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200) |
| **ID / tipo** | `ESA/WorldCover/v200` · `ImageCollection` |
| **Resolución espacial** | 10 m |
| **Período de referencia** | Año 2021 (mapa único; la versión v100 corresponde a 2020) |
| **Banda** | `Map` (código de clase: 10, 20, …, 100) |
| **Procesamiento** | `filterBounds(AOI)` → `mosaic()` → recorte. Visualización categórica que **conserva los códigos originales**. Área por clase con `ee.Image.pixelArea()` agrupada por clase. |
| **Unidades** | Original: código de clase. Final: código de clase (mapa) y hectáreas / % (tabla). |
| **Licencia** | CC-BY-4.0 |

> **Ojo al comparar 2020 vs 2021:** v100 y v200 usan versiones distintas del algoritmo, así que las diferencias mezclan cambios reales y cambios metodológicos. No es un producto para detectar cambio de cobertura.

In [ ]:
res_lc = visualize_from_registry("landcover_worldcover", AOI)
display(res_lc.summary())
res_lc.map

In [ ]:
# Área por clase en el AOI (cómputo en el servidor; viaja solo una tabla pequeña).
# A 10 m el cálculo es exacto pero más lento; para AOIs grandes usa scale=30 o más.
lc_areas = compute_class_areas(res_lc.image, "Map", AOI, scale=10, class_info=WORLDCOVER_CLASSES)
lc_areas

**Interpretación.** La leyenda asocia cada código con su clase. En el AOI por defecto se espera **área construida** concentrada en la zona urbana de Cali, **cultivos y herbazales** en la planicie del río Cauca (al oriente) y **cobertura arbórea** dominante hacia la cordillera Occidental y los Farallones (al occidente). La tabla de áreas resume la composición del paisaje; recuerda que la exactitud global reportada del producto es cercana al 75–77 %, así que las clases espectralmente parecidas (p. ej. herbazal vs. cultivo) pueden confundirse a escala local.

<a id="10-ejemplo-2-dem"></a>
## 10. Ejemplo 2 — Modelo Digital de Elevación (NASA SRTM)

| Ítem | Detalle |
|---|---|
| **Fuente** | NASA / USGS — Shuttle Radar Topography Mission (SRTM), versión 3 con relleno de vacíos. |
| **Enlace GEE** | [USGS/SRTMGL1_003](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003) |
| **ID / tipo** | `USGS/SRTMGL1_003` · `Image` |
| **Resolución espacial** | 1 arco-segundo (~30 m) |
| **Resolución temporal** | Única adquisición (febrero de 2000). Cobertura ~60° N – 56° S. |
| **Banda** | `elevation` |
| **Procesamiento** | Selección de banda → recorte → estiramiento por percentiles 2–98 → estadísticas en el AOI. Pendiente con `ee.Terrain.slope()` (grados). |
| **Unidades** | Original: m. Final: m (elevación) y grados (pendiente). |

Alternativa: Copernicus DEM GLO-30 (`COPERNICUS/DEM/GLO30`, banda `DEM`), que es una colección de teselas; requiere `mosaic()` y fijar la proyección antes de calcular pendientes.

In [ ]:
res_dem = visualize_from_registry("dem_srtm", AOI, compute_stats=True)
display(res_dem.statistics.round(1))
res_dem.map

In [ ]:
# Capa opcional de pendiente (grados). ee.Terrain.slope usa la proyección nativa de SRTM.
INCLUDE_SLOPE = True

if INCLUDE_SLOPE:
    slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003")).clip(AOI)  # banda 'slope'
    slope_vis = {"min": 0, "max": 45, "palette": PALETTES["slope"]}
    res_slope = GEEResult(image=slope, vis_params=slope_vis, layer_name="Pendiente SRTM (°)",
                          metadata={"dataset_id": "USGS/SRTMGL1_003", "band": "slope"}, scale=30,
                          colorbar_label="Pendiente (grados)")
    res_slope.statistics = compute_statistics(slope, "slope", AOI, scale=30)
    display(res_slope.statistics.round(1))

    hillshade = ee.Terrain.hillshade(ee.Image("USGS/SRTMGL1_003")).clip(AOI)
    res_dem.map.add_layer(hillshade, {"min": 0, "max": 255}, "Sombreado (hillshade)", False)
    add_result_to_map(res_dem.map, res_slope, shown=False, legend_position="bottomleft")

res_dem.map

**Interpretación.** La tabla muestra mínimo, máximo, media y desviación estándar de la elevación dentro del AOI (calculadas a 30 m en el servidor). En el AOI por defecto se observa el contraste entre el **valle del río Cauca** (la cota más baja, cerca de 1000 m) y la **cordillera Occidental**, que en los Farallones supera holgadamente los 3000 m. La pendiente resalta las vertientes empinadas de la cordillera frente a la planicie casi plana. Activa *Sombreado* y *Pendiente* desde el control de capas.

<a id="11-ejemplo-3-precipitacion"></a>
## 11. Ejemplo 3 — Precipitación (CHIRPS Daily)

| Ítem | Detalle |
|---|---|
| **Fuente** | Climate Hazards Center (UC Santa Barbara). CHIRPS combina imágenes satelitales infrarrojas con datos de estaciones. Versión 2.0 *Final*. |
| **Enlace GEE** | [UCSB-CHG/CHIRPS/DAILY](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY) |
| **ID / tipo** | `UCSB-CHG/CHIRPS/DAILY` · `ImageCollection` (1 imagen por día) |
| **Resolución espacial** | 0,05° (~5,5 km; 5566 m en GEE). Cobertura cuasi-global 50° S – 50° N. |
| **Resolución temporal** | Diaria, desde 1981. El producto *Final* se publica con rezago de varias semanas; revisa la disponibilidad en el catálogo. |
| **Banda** | `precipitation` |
| **Procesamiento** | `filterBounds` + `filterDate([START_DATE, END_DATE))` → **`sum()`** (acumulado) o **`mean()`** (promedio diario) → recorte → estadísticas. |
| **Unidades** | Original: **mm/d**. Acumulado: **mm** en el período. Promedio: **mm/d**. |

### Acumulación vs. promedio temporal

- **Acumulado (`sum`)**: suma de la lámina diaria sobre los *N* días → **mm totales** del período. Es lo que se usa para totales mensuales, estacionales o anuales. Su magnitud **depende de la longitud del período**: un acumulado de 3 meses no es comparable con uno de 12.
- **Promedio (`mean`)**: media de la lámina diaria → **mm/d**. Es independiente de la longitud del período, útil para comparar períodos de distinta duración.
- Relación: `acumulado (mm) = promedio (mm/d) × N días` (si no faltan días). El código verifica que el número de imágenes coincida con el de días; si faltan, el acumulado queda **subestimado**.

In [ ]:
res_pr_sum = visualize_from_registry("precip_chirps_sum", AOI, compute_stats=True)

expected = n_days(START_DATE, END_DATE)
got = res_pr_sum.metadata["n_images"]
print(f"Imágenes diarias usadas: {got} de {expected} días del período.")
if got < expected:
    print("⚠ Faltan días: el acumulado está subestimado. Revisa la disponibilidad de CHIRPS o acorta el período.")

display(res_pr_sum.summary())
display(res_pr_sum.statistics.round(1))
res_pr_sum.map

In [ ]:
# Promedio diario del mismo período (mm/d) en el mismo mapa, como capa adicional.
res_pr_mean = visualize_from_registry("precip_chirps_mean", AOI, compute_stats=True, show_map=False)
add_result_to_map(res_pr_sum.map, res_pr_mean, shown=False, legend_position="bottomleft")
display(res_pr_mean.statistics.round(2))

# Chequeo de consistencia: media espacial del acumulado ≈ media espacial del promedio × N días
check = res_pr_mean.statistics["media"].iloc[0] * got
print(f"Promedio × N = {check:,.1f} mm  vs.  acumulado medio = {res_pr_sum.statistics['media'].iloc[0]:,.1f} mm")
res_pr_sum.map

**Interpretación.** El mapa de acumulado muestra la lámina total del período; la barra de colores está en **mm**. La capa de promedio (actívala en el control de capas) muestra la misma información normalizada por día (**mm/d**). La comparación numérica de la última celda debe coincidir (salvo redondeo y diferencias de remuestreo) si no faltan días. Con ~5,5 km de píxel, CHIRPS capta el patrón regional (por ejemplo, el contraste entre la vertiente occidental de la cordillera y el valle interandino), pero **no** gradientes locales finos; además, en zonas montañosas con pocas estaciones su incertidumbre es mayor.

<a id="12-ejemplo-4-temperatura"></a>
## 12. Ejemplo 4 — Temperatura (ERA5-Land Daily Aggregates)

| Ítem | Detalle |
|---|---|
| **Fuente** | ECMWF / Copernicus Climate Change Service. ERA5-Land es un **reanálisis** (modelo + observaciones) de la componente terrestre de ERA5. Agregados diarios preparados por Google a partir de los datos horarios. |
| **Enlace GEE** | [ECMWF/ERA5_LAND/DAILY_AGGR](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR) |
| **ID / tipo** | `ECMWF/ERA5_LAND/DAILY_AGGR` · `ImageCollection` |
| **Resolución espacial** | 0,1° (~11 km; 11 132 m en GEE) |
| **Resolución temporal** | Diaria desde 1950, con rezago aproximado de 3 meses respecto al tiempo real. |
| **Banda** | `temperature_2m` (temperatura del aire a 2 m; en los agregados diarios es la **media** de las 24 horas) |
| **Procesamiento** | `filterBounds` + `filterDate` → `mean()` (media del período) → **K → °C** (`− 273,15`) → recorte → estadísticas. |
| **Unidades** | Original: **K**. Final: **°C**. |

> **Resolución vs. AOI.** Con píxeles de ~11 km, un AOI de ~45 × 40 km contiene pocos píxeles. El recorte muestra los píxeles que intersecan el AOI y las estadísticas usan los centros de píxel que caen dentro. ERA5-Land suaviza el relieve: en montaña la temperatura de la celda corresponde a la elevación media de la celda, no a la del punto.

In [ ]:
res_t = visualize_from_registry("temp_era5land", AOI, compute_stats=True)
display(res_t.summary())
display(res_t.statistics.round(2))
res_t.map

**Interpretación.** El mapa muestra la temperatura media del aire a 2 m del período en **°C**. Se espera un gradiente asociado a la altitud: celdas más cálidas sobre el valle del río Cauca y más frías sobre la cordillera Occidental (en promedio la temperatura disminuye del orden de 0,6 °C por cada 100 m). Compara con el DEM del ejemplo 2 en el mapa combinado de la siguiente sección.

<a id="13-mapa-final-combinado"></a>
## 13. Mapa final combinado y comparación

Todas las capas comparten el **mismo AOI**. Usa el control de capas (arriba a la derecha) para activarlas y desactivarlas, ajusta la transparencia desde la barra de herramientas de `geemap` y usa el **inspector** para consultar valores de píxel de cada capa en un punto.

Las capas no se descargan: `geemap` solicita a GEE teselas renderizadas según el zoom.

In [ ]:
m_all = create_map()
add_result_to_map(m_all, res_lc, shown=True, opacity=0.8, legend_position="bottomright")
add_result_to_map(m_all, res_dem, shown=False, legend_position="bottomleft")
add_result_to_map(m_all, res_pr_sum, shown=False, legend_position="bottomleft")
add_result_to_map(m_all, res_t, shown=False, legend_position="bottomleft")
add_aoi_outline(m_all, AOI)
m_all.center_object(AOI)
if hasattr(m_all, "add_inspector"):  # disponible en versiones recientes de geemap
    m_all.add_inspector(position="topright", opened=False)
m_all

### Comparación lado a lado (split map)

Útil para contrastar dos variables en la misma ubicación, por ejemplo cobertura vs. elevación o temperatura vs. elevación. Cambia `left` y `right` por cualquiera de los resultados (`res_lc`, `res_dem`, `res_pr_sum`, `res_pr_mean`, `res_t`).

In [ ]:
left, right = res_lc, res_dem

m_split = geemap.Map(height=MAP_HEIGHT, basemap=BASEMAP)
m_split.split_map(
    left_layer=geemap.ee_tile_layer(left.image, left.vis_params, left.layer_name),
    right_layer=geemap.ee_tile_layer(right.image, right.vis_params, right.layer_name),
    left_label=left.layer_name,
    right_label=right.layer_name,
)
m_split.center_object(AOI)
m_split

In [ ]:
# Tabla resumen de estadísticas de las variables continuas (misma área de estudio)
summary_stats = pd.concat({
    "Elevación (m)": res_dem.statistics,
    "Precipitación acumulada (mm)": res_pr_sum.statistics,
    "Precipitación media (mm/d)": res_pr_mean.statistics,
    "Temperatura media (°C)": res_t.statistics,
}).droplevel(1)
summary_stats.round(2)

<a id="14-exportacion"></a>
## 14. Exportación (opcional)

Desactivada por defecto (`RUN_EXPORTS = False`). Al activarla se crean tareas batch hacia la carpeta `EXPORT_FOLDER` de tu Google Drive. Las tareas consumen cuota de cómputo del proyecto (EECU), así que exporta solo lo que necesites y con la escala adecuada a cada dataset.

In [ ]:
if RUN_EXPORTS:
    tasks = [
        export_image_to_drive(res_dem.image, "srtm_elevacion_aoi", AOI, scale=30, start=True),
        export_image_to_drive(res_pr_sum.image, f"chirps_acum_{START_DATE}_{END_DATE}", AOI, scale=5566, start=True),
        export_image_to_drive(res_t.image, f"era5land_t2m_media_{START_DATE}_{END_DATE}", AOI, scale=11132, start=True),
    ]
    for t in tasks:
        print(t.status()["description"], "→", t.status()["state"])
else:
    print("Exportación desactivada (RUN_EXPORTS = False).")

# Descarga directa de un recorte pequeño (sin Drive). Descomenta para usar:
# download_small_geotiff(res_t.image, "outputs/era5land_t2m_media.tif", AOI, scale=11132)

<a id="15-como-agregar-un-dataset-nuevo"></a>
## 15. Cómo agregar un dataset nuevo

**Opción A — llamada directa (sin tocar el registro):**

```python
res_ndvi = visualize_gee_dataset(
    dataset_id="MODIS/061/MOD13Q1",      # verifica el ID y la banda en el catálogo
    band="NDVI",
    aoi=AOI,
    start_date="2024-01-01", end_date="2025-01-01",
    reducer="median",
    unit_conversion=lambda img: img.multiply(0.0001),   # factor de escala del producto
    vis_params={"min": 0, "max": 0.9, "palette": ["ffffe5", "78c679", "004529"]},
    colorbar_label="NDVI (–)",
    compute_stats=True,
)
res_ndvi.map
```

**Opción B — agregar una ficha al `DATASET_REGISTRY`** (recomendado si se va a reutilizar):

```python
DATASET_REGISTRY["ndvi_modis"] = {
    "title": "MODIS NDVI 250 m (mediana)",
    "dataset_id": "MODIS/061/MOD13Q1", "dataset_type": "ImageCollection", "band": "NDVI",
    "reducer": "median", "temporal": True,
    "unit_conversion": lambda img: img.multiply(0.0001),
    "vis_params": {"min": 0, "max": 0.9, "palette": ["ffffe5", "78c679", "004529"]},
    "colorbar_label": "NDVI (–)", "units": ("NDVI × 10000", "NDVI (–)"),
    "resolution": "250 m; 16 días",
    "url": "https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13Q1",
}
res_ndvi = visualize_from_registry("ndvi_modis", AOI, compute_stats=True)
```

> Antes de agregar un dataset revisa en su página del catálogo: ID exacto, tipo (`Image`/`ImageCollection`), nombre de banda, **unidades y factor de escala**, resolución, disponibilidad temporal y licencia. El código valida ID, banda y período, pero no puede adivinar factores de escala.

<a id="16-problemas-frecuentes"></a>
## 16. Problemas frecuentes

| Síntoma / mensaje | Causa probable | Solución |
|---|---|---|
| `Configura PROJECT_ID…` | No cambiaste el valor por defecto. | Pon tu ID en la sección 5 (o en el secreto `EE_PROJECT_ID`). |
| `…not registered to use Earth Engine` | Proyecto sin registro EE. | Regístralo en la página de Earth Engine de la Cloud Console (guía, paso 6). |
| `…API has not been used in project… / SERVICE_DISABLED` | API deshabilitada. | Habilita `earthengine.googleapis.com` (guía, paso 5). |
| `Caller does not have required permission to use project` | Falta `serviceusage.services.use`. | Rol **Service Usage Consumer** en el proyecto. |
| `PERMISSION_DENIED` al leer un asset | Sin rol de EE o asset privado. | Rol **Earth Engine Resource Viewer** (o permisos sobre el asset). |
| Errores de credenciales / `invalid_grant` | Credenciales vencidas o de otra cuenta. | `initialize_ee(PROJECT_ID, force_auth=True)`. |
| El mapa no aparece en Colab | Gestor de widgets no habilitado. | Ejecuta la celda de instalación (habilita `enable_custom_widget_manager`) y reinicia la sesión si persiste. |
| `GEEBandError` | Nombre de banda incorrecto. | El mensaje lista las bandas disponibles. |
| `GEEEmptyCollectionError` | Período sin datos (p. ej. más reciente que el rezago del producto) o AOI fuera de cobertura. | El mensaje indica el rango disponible; ajusta fechas o AOI. |
| `User memory limit exceeded` / `Computation timed out` | Consulta interactiva demasiado pesada. | Reduce AOI o período, aumenta `scale` o exporta en batch. |
| `Too many concurrent aggregations` / 429 | Límite de concurrencia o cuota. | Espera y reintenta; revisa el nivel de cuota no comercial del proyecto. |
| Descarga directa falla por tamaño | Límite de `getDownloadURL`. | Usa `export_image_to_drive()` o aumenta `scale`. |

<a id="17-rendimiento-y-buenas-practicas"></a>
## 17. Rendimiento y buenas prácticas

- **Filtra primero, calcula después:** `filterBounds` y `filterDate` antes de cualquier agregación reducen drásticamente el trabajo del servidor.
- **`getInfo()` solo para cosas pequeñas:** escalares, listas cortas, diccionarios de estadísticas. Nunca sobre imágenes o colecciones completas.
- **Visualiza con teselas, no descargando:** `Map.add_layer` usa los servicios de GEE; la descarga solo tiene sentido para análisis fuera de GEE.
- **Usa una escala coherente con el dataset:** calcular estadísticas de ERA5-Land a 10 m no agrega información y multiplica el costo.
- **`bestEffort=True` y `tileScale`:** en `reduceRegion` evitan errores de memoria en AOIs grandes (a costa de, posiblemente, una escala más gruesa en `bestEffort`).
- **Exportaciones batch** para resultados grandes o finos; las consultas interactivas tienen límites de tiempo y memoria.
- **Cuotas no comerciales:** desde abril de 2026 los proyectos no comerciales tienen un cupo mensual de EECU-hora (Community/Contributor/Partner). Ver `GOOGLE_CLOUD_SETUP.md`.
- **Reproducibilidad:** fija fechas explícitas, versiones de datasets (p. ej. `v200`) y versiones de paquetes (`requirements.txt`). Los productos en tiempo casi real (CHIRPS, ERA5-Land) pueden reprocesarse.
- **Seguridad:** no incluyas credenciales, tokens ni claves en el notebook ni en el repositorio (el `.gitignore` excluye los patrones habituales).

<a id="18-referencias"></a>
## 18. Referencias

**Datasets (catálogo oficial de Earth Engine)**
- ESA WorldCover 10 m v200 — https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200 · Zanaga et al. (2022), doi:10.5281/zenodo.7254221
- NASA SRTM Digital Elevation 30 m — https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003
- Copernicus DEM GLO-30 — https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30
- CHIRPS Daily v2.0 — https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY · Funk et al. (2015), *Scientific Data* 2, 150066, doi:10.1038/sdata.2015.66
- ERA5-Land Daily Aggregates — https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_DAILY_AGGR · Muñoz-Sabater et al. (2021), *ESSD* 13, 4349–4383, doi:10.5194/essd-13-4349-2021
- FAO GAUL 2025 (nivel 1 y 2) — https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2025_level2

**Earth Engine, Cloud y Colab**
- Autenticación e inicialización — https://developers.google.com/earth-engine/guides/auth
- Configuración de proyectos Cloud para EE — https://developers.google.com/earth-engine/cloud/earthengine_cloud_project_setup
- Control de acceso y roles IAM — https://developers.google.com/earth-engine/guides/access_control
- Niveles de cuota no comercial — https://developers.google.com/earth-engine/guides/noncommercial_tiers
- Cliente vs. servidor — https://developers.google.com/earth-engine/guides/client_server
- Instalación del cliente Python — https://developers.google.com/earth-engine/guides/python_install
- geemap — https://geemap.org
- Google Colab — https://colab.research.google.com